# Data Science Case Study: Candidate Resume Search Platform

## Business Context & Objective

### Background
You are joining the Business Development team at Millennium which is a global hedge fund that manages assets across multiple investment strategies (fundamental equity, systematic trading, credit, etc.). The BD team is responsible for sourcing junior analyst talent across different:

- **Geographic Markets**: US, Europe, Asia-Pacific
- **Investment Approaches**: Fundamental vs. Systematic/Quantitative strategies
- **Sectors**: Technology, Healthcare, Financial Services, Energy, Industrials, Consumer, Credit, Macro, etc.
- **Experience Levels**: depending on the job requisitions

### Goal
Build a **searchable platform** to quickly identify candidates based on specific criteria based on job requisitions.

### Your Task
1. Parse resume data from PDF/Word documents using **LLM models via API**
2. Create parsed resume data as **JSON, CSV, etc.** for further analysis
3. Create a **Streamlit web application** where BD users can search and filter candidates using multiple criteria
   (Based on the background provided above, you will come up with relevant filters)
4. Visualize candidate distributions and insights
5. **Design for scalability** to handle large volumes of resumes

### Output
1. Code for data parsing and Streamlit in **this Jupyter notebook** for ease of review
2. Include the **link to Streamlit app** in the notebook
3. **JSON/CSV exports** of parsed resume data
4. Discussion of additional features and implementation approach if more time was available
---

## Sample Resume Data

**You have access to 10 made up resume files representing different candidate profiles**

---

## Technical Requirements & Evaluation
  
### Evaluation Criteria
1. **Technical Implementation (40%)**: Code quality, data processing, performance
2. **User Experience (30%)**: Interface design, functionality, responsiveness
3. **Business Value (20%)**: Feature completeness, search capabilities, insights
4. **Documentation (10%)**: Code comments, README, presentation

### Time Recommendation
- **Total Estimated Time**: ~9 hours (**tight but achievable with focused approach**)
- **Recommended Strategy**: Build core MVP in 6-7 hours, then enhance

### Phased Approach (Recommended)
**Phase 1 (3 hours): Core Parsing & Data**
- Set up LLM API integration (OpenAI, Anthropic, or alternatives)
- Parse 2-3 sample resumes using LLM
- Create structured data format with JSON, etc.
- Build basic data validation and quality checks
 
**Phase 2 (3 hours): Streamlit**  
- Basic filter interface
- Simple results display
- Core visualizations

**Phase 3 (2-3 hours): Enhancement**
- Parse remaining resumes
- Add advanced filters
- Polish UI/UX

### Success Criteria (MVP)
- LLM-powered parsing of at least 5-7 resume files with high accuracy
- Working Streamlit app with basic search functionality
- At least 3-4 meaningful filters working
- A few visualizations showing candidate insights
- JSON/CSV output of parsed resume data
- Clean, documented code


---

## Solution summary

### Repository goal

Take a folder of resumes and turn it into a searchable candidate database for a hedge fund BD team. An LLM reads each DOCX or PDF into a structured record, code derives the filterable facts from that record, SQLite stores and indexes it, and a Streamlit app lets a BD user describe a requisition and get back a ranked, explained shortlist. The same pipeline runs three ways: from the CLI, from the app's Ingest page, and from the submission notebook at the repository root, which writes every source file out with `%%writefile` and runs the demos.

The 10 sample resumes in `data/` are the working set. Every number quoted below comes out of the eval result files in `evals/results/` and the `parse_jobs` table, both produced by code in this repository.

### Repository layout

```
2025_ds_case_study_resume_platform.ipynb   submission notebook (generated by scripts/build_notebook.py)
README.md                                  quick start, CLI reference, measured results
docs/design.md                             design record: schema, prompt, verification table, scale-up notes
pyproject.toml                             uv project, Python 3.12
taxonomy/                                  YAML: countries and regions, sectors, firm types, asset classes,
                                           skill synonyms, ranking weights
src/resume_search/
  extract/     docx.py pdf.py repair.py quality.py types.py   file -> ExtractedText
  parse/       models.py prompt.py client.py verify.py        ExtractedText -> ParsedResume (3 LLM calls + checks)
  derive/      experience.py location.py skills.py models.py  ParsedResume -> DerivedFields, in code
  store/       schema.sql db.py fts.py export.py              SQLite upsert, FTS body, JSON/CSV exports
  search/      requisition.py rules.py keyword.py vectors.py rank.py weights.py llm.py
  ingest/      pipeline.py jobs.py                            ingest() and the parse_jobs state machine
  eval/        judge.py metrics.py parsing_eval.py ranking_eval.py report.py results.py
  taxonomy.py cli.py
app/           Home.py common.py pages/1_Search.py 2_Candidate.py 3_Insights.py 4_Ingest.py 5_Evals.py
evals/         requisitions.jsonl (6 requisitions), gold/ (hand grades), results/ (judge runs)
exports/       candidates.json candidates.csv parsed/ text/   (candidates.sqlite is git-ignored)
scripts/       build_notebook.py notebook_text.py build_review_sheet.py
tests/         pytest suite
```


### Technical detail: parsing

**Extraction.** `python-docx` walks the document body in order, so a header that happens to live in a table stays at the top where it belongs, and each table row becomes one line with cells joined by ` | `. PyMuPDF reads PDF pages as text. A repair map swaps out ligature glyphs (`Ɵ` standing in for `ti`, `ﬁ`, `ﬂ`, and friends), then an unknown-word ratio against an English wordlist plus the taxonomy terms tells you how much damage is left. Fewer than 400 characters on a PDF page, an image over half a page, or more than 10% unknown words after repair, and the file goes down the vision path instead: pages rendered to PNG, sent as image blocks, same schemas.

**Three constrained-decoding calls.** The full record is bigger than the structured-output grammar allows, so parsing is three calls. Each uses `client.messages.parse(..., output_format=Model)`, which means the model can only emit JSON that validates against the Pydantic schema. No free-form output to clean up afterwards.

| Call | Input | Output | Runs |
|---|---|---|---|
| profile | filename + resume text | name, contact, education, skills, certifications, languages | in parallel with experience |
| experience | resume text | list of roles, each with per-stint dates, type, relevance, firm type, sectors with evidence quotes | in parallel with profile |
| assessment | the verified profile and experience as JSON, plus the text | investment approach with evidence quote, 60–90 word `search_summary` | after cross-verification |

**Prompt rules.** Copy, never infer. If the resume does not say it, the field is null. Every judgement call (sector, approach, markets covered) carries a verbatim quote, and a quote that is not actually a substring of the text fails validation. Dates are `YYYY-MM` or `YYYY`; a role worked in several separate periods is several stints, not one span. Investment approach is `Fundamental`, `Systematic`, or `Both` only when the evidence comes from a full-time core or adjacent role. Otherwise `Unclear`. The `search_summary` has six fixed slots (seniority and approach, assets and sectors, markets, current employer and title, standout skills, degree and certifications), no contact details, no adjectives.

**Retry ladder.** Each call climbs Sonnet, then Sonnet with the validation errors appended, then Opus with the errors. Every attempt records model, tokens, latency, and error, and the `parse_jobs` row adds them up into cost and latency.

**Derivation in code.** Months of experience merge the date ranges of every full-time relevant stint, so two overlapping jobs count once. Roles with no dates use the duration text. Seniority band falls out of those months. Country and region come from the taxonomy table, contact block first, latest located role or school as the fallback. Skills go through the synonym dictionary with rapidfuzz at score 90 behind it. CFA status is a regex over the certifications. None of this asks the LLM anything.


### Technical detail: verification

`parse/verify.py` is a registry with one check per field. A check names a deterministic extractor over the repaired text and filename, a comparator against the LLM's value, and the smallest schema to regenerate when the two disagree. Adding a check is adding a row.

| Field | Deterministic source | Comparator |
|---|---|---|
| name | filename stem minus `Resume`, `CV`, `CFA`, dates; first non-empty line | token-set ratio ≥ 80 |
| email, phone | regexes | exact; digits-only |
| certifications | CFA, FRM, CAIA, CPA, ACCA and level or candidacy mentions in text and filename | set equality on canonical tokens |
| skills | taxonomy term matcher over the text | every matched term must appear in the LLM list; LLM extras allowed |
| stint dates | every `YYYY-MM` or `YYYY` the LLM emits must match a month-year or year token in the text | all grounded or not |
| duration | `N years M months` near the employer line, and the stint sum | within 1 month |
| role count | employer lines from the known-firm list and title patterns | LLM roles ≥ count |
| degree level | degree keyword regex (MBA, MSc, BSc, PhD, PGDM, Diplôme d'Ingénieur) | mapped level equals LLM level |
| country | phone prefix and location-alias gazetteer | equality |
| investment approach | which role contains the evidence quote | label is `Unclear`, or the role is full-time core or adjacent |

The policy. Run every check once the profile and experience calls are back. For each disagreement, make one targeted regeneration call whose schema holds only that field, handing the model the text, its own previous value, and the deterministic finding stated as evidence ("the text contains 'CFA' at ...; your certifications list was empty"). Re-run the check. Keep the regeneration only if it fixed the disagreement; otherwise keep the LLM value and record the pair under `verification.unresolved`. The assessment call gets the verified profile and experience, so the summary is written from checked facts, and its own approach check runs afterwards with one regeneration of its own. The whole `meta.verification` block (`checks_run`, `disagreements`, `regenerated`, `unresolved`, `details`) is stored with the record and shown on the Evals page.


### Technical detail: search ranking

A `Requisition` comes from one of three places: the filter widgets, a pasted job description that one Sonnet call turns into a draft the user then edits, or CLI flags. Every criterion carries a mode, `must` or `preferred`.

1. **Prune.** Each `must` criterion becomes a SQL predicate over the `candidates` and facet tables. The years range joins in when its mode is `must`. Only survivors get scored.
2. **Rule score.** Each `preferred` criterion the user set contributes `weight × match`, and the sum is divided by the sum of the weights that were set. Weights live in `taxonomy/weights.yaml`: approach 3, sector 3, location 2, markets covered 2, years in range 2, each preferred skill 1, degree 1, certification 1, asset class 1. Match rules: `Both` matches either approach; sectors score the fraction requested that match; years score 1 inside the range and decay linearly to 0 at 2 years outside; skills match on canonical token or rapidfuzz ≥ 90; degree matches at or above the requested level. Location depends on scope: a current-country match scores 1.0, a worked-only match 0.5, and a region-level match where a country was asked scores half of either.
3. **Keyword score.** FTS5 BM25 over the free text plus required and preferred skills, against a body that is the resume text with the contact block cut out. Min-max scaled over the pruned set.
4. **Embedding score.** Cosine between the `bge-small-en-v1.5` embedding of the free text and the candidate's vectors, one for the summary and one per role. The candidate takes the max. Min-max scaled over the pruned set.
5. **Final.** `0.5 × rule + 0.2 × keyword + 0.3 × embedding`. With no free text, keyword and embedding drop out and the rule score is renormalized. Ties go to name order.
6. **Optional rerank.** A toggle sends the top 10 with the requisition and each summary to Sonnet, which hands back a new order and a one-line reason each. If that call fails you get the plain ranking.

Each result card shows one chip per criterion and keyword, matched or not, with the matching value spelled out ("Worked in GB 4.0y"). A score breakdown expander shows the three components and every rule term, so you can see why someone landed where they did.


### Technical detail: database design

SQLite, one file, created from `store/schema.sql` on connect.

| Table | Purpose |
|---|---|
| `candidates` | one row per candidate: scalar filter columns (country, region, months of experience, seniority band, approach, highest degree, CFA), the three contact columns, the `search_summary`, and `record_json` holding the full `meta` + `extracted` + `derived` record |
| `experience`, `education` | one row per role or degree, with first start, last end, months, and the stint list as JSON |
| `candidate_sectors`, `candidate_tags`, `candidate_assets`, `candidate_skills`, `candidate_countries`, `candidate_certs`, `candidate_languages` | many-to-many facets, one row per (candidate, value); sectors carry their evidence quote, countries carry months and a `worked` or `covered` kind |
| `candidate_fts` | FTS5 virtual table (porter, unicode61) over the resume text minus the contact block |
| `embeddings` | float32 little-endian blobs, one per (candidate, kind, idx) with the model name and dimension |
| `parse_jobs` | one row per ingest job: file hash, prompt version, status, attempts JSON, tokens, cost, latency, error, timestamps |
| `taxonomy_country`, `taxonomy_sector`, `taxonomy_skill_synonym`, `ranking_weights` | the YAML taxonomy mirrored into tables at startup |

A few rules hold the design together. The scalar columns and facet tables exist for filters and are indexed on country and months; `record_json` is the source of truth and doubles as the JSON export. Contact fields live in their own columns and never reach the FTS body or the embeddings. Each candidate is written in one transaction that deletes its child rows and reinserts them, so a failed ingest leaves the previous version intact. The idempotency key is `(file_sha256, prompt_version)`: bump the prompt and you re-parse; bump only the schema version and `rederive` rebuilds `derived` from `record_json` with no API call. `exports/candidates.json` and `candidates.csv` get rewritten at the end of every ingest.


### Evaluation process

The judge is always a different, stronger model than the one doing the work. Sonnet parses, Opus grades. Results land in `evals/results/<suite>-<timestamp>.json`, and both the Evals page and the submission notebook read the latest file per suite.

**Parsing suite.** For each resume the judge gets the repaired text and the `extracted` JSON and grades every top-level field (name, contact, education, experience, skills, certifications, languages, investment approach, search summary) plus every experience entry as `correct`, `partial`, `wrong`, `hallucinated`, or `missing`, with a one-line reason. Metrics are accuracy (`correct` over total), soft accuracy (`correct` plus `partial`), and the hallucinated and missing counts, per field and overall. I hand-graded three resumes chosen for their shapes (a clean DOCX, a table-heavy DOCX, a PDF with ligature glyphs) in `evals/gold/parsing_human.jsonl`; the judge's exact and within-one-level agreement with those 42 grades is reported, and the judge prompt itself is versioned (`pj3` in the latest run). A second gold file, `evals/gold/location_human.jsonl`, checks derived `current_country` for all 10 candidates. `scripts/build_review_sheet.py` lays my grades and the judge's side by side in `evals/gold/REVIEW.md`.

**Ranking suite.** Six requisitions in `evals/requisitions.jsonl`: US fundamental healthcare, APAC systematic equities, Europe credit, India sell-side healthcare, junior quant developer, senior generalist long/short. For each one the judge grades every candidate 0, 1, or 2 from the derived facets and the summary, each candidate on its own with no comparison to the others. Then the suite runs the search four ways, full blend plus the three single-signal ablations (rule only, BM25 only, embedding only), and reports nDCG@5 and P@3 per requisition and on average. Grades are cached by requisition and candidate content hash, so a re-run only grades what changed.

**Latest results.**

| Measure | Value |
|---|---|
| Parsing accuracy | 97.9% of 143 fields `correct`, 99.0% `correct` or `partial`, 0 hallucinated, 0 missing |
| Judge against human | 95.2% exact, 100% within one level, 42 fields |
| Location | 10 of 10 |
| Ranking, mean nDCG@5 | full blend 0.969, rule only 0.958, BM25 only 0.968, embedding only 0.927 |
| Ranking, mean P@3 | full blend 0.722, rule only 0.722, BM25 only 0.778, embedding only 0.722 |
| Cost | $0.078 per parsed resume; $0.65 per parsing eval; $0.17 per ranking eval |

**How the evals get used.** Any prompt or schema change gets `resume-search eval run --suite parsing` before and after. Compare per-field accuracy, and check whether the judge-human agreement moved as well. A change that costs a field accuracy or adds a hallucination shows up before it ever reaches the index. That loop is how the prompt in `parse/prompt.py` went from v1 to v7.


### Scalability design

**Ingest as a job queue.** `ingest()` already treats each file as an independent job. Its `parse_jobs` row only ever moves forward, `queued`, `extracting`, `parsing`, `deriving`, `indexing`, and ends at `ok`, `failed`, or `skipped`. The idempotency key `(file_sha256, prompt_version)` lets a re-run skip finished work, and each candidate is one transaction. The 4-thread pool is the scheduler for now. At volume the same rows become a queue (SQS, Cloud Tasks, or a Postgres `SKIP LOCKED` table) with stateless workers claiming rows, and neither the state machine nor the key changes. At the measured 28.4 seconds a resume, 100,000 resumes want about 100 concurrent workers for an 8-hour load, plus a token-bucket limiter in front of the API client sized to the account's rate limit.

**Storage.** The DDL moves to Postgres with three edits: FTS5 becomes a `tsvector` column with a GIN index, the `embeddings` table becomes a `pgvector` column with an HNSW index in place of the in-memory cosine scan, and `parse_jobs` becomes the work queue. Keeping `extracted` and `derived` apart is what keeps schema changes cheap: a derive change re-runs over stored records at zero API cost, and only a prompt change re-parses.

**Cost.** $0.078 a resume is about $7,800 per 100,000 at list prices. Prompt caching is already on for the system prompts. The batch API suits an overnight bulk load at a discount. And a TF-IDF plus logistic-regression classifier trained on the LLM's own `investment_approach` and sector labels, checked with a confusion matrix against them, handles most resumes for almost nothing and only sends the low-confidence ones to the LLM.

**Inputs.** Text, markdown, and HTML read directly. doc, rtf, odt, and pages go through LibreOffice headless to DOCX or PDF. Scanned PDFs and images take the vision path or Tesseract OCR. LinkedIn JSON and ATS XML map straight into `extracted` and skip the LLM altogether.

**Term matching.** Two layers on top of the synonym dictionary: a spaCy EntityRuler compiled from the taxonomy, giving an independent skill and country extraction and highlighting evidence on the Candidate page, and a networkx knowledge graph over the taxonomy that expands a query for "systematic" into "stat arb", "factor", and "quant".

**PII.** Contact fields stay in their own columns and out of the FTS body and the embeddings; a validator rejects a summary carrying an `@` or a seven-digit run. Two things before production: strip the contact block before the API call, and a read-time redaction switch that masks the contact columns by user role.

**Serving.** Streamlit caches the connection, the embedding model, and the taxonomy as resources, caches query results by requisition hash, and clears all of it after an ingest. With many users the search function moves behind an API process and the app becomes a thin client. The ranking code is already a pure function of a connection, a requisition, and the weights, so nothing has to be untangled first.
